# DS-ST K-Fold — NOC Prediction (GF Kit, GroupKFold)

GroupKFold(5) · pretrained backbone (noc_10mb) · NOC fine-tune (FocalLoss + oversampling)

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'sympy==1.13.3', '-q'])

import os, gc, time, json, warnings, io, zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from collections import Counter
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from torch.utils.data import DataLoader, TensorDataset, Dataset

warnings.filterwarnings('ignore')

CSV_PATH  = '/kaggle/input/datasets/nguyenmanhhust/gf-kit-file/gf_groups.csv'
CKPT_PATH = '/kaggle/input/datasets/nguyenmanhhust/noc-10mb/noc_10mb'
sys.path.insert(0, '/kaggle/input/datasets/mashwoo/set-transformer-10mb')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
N_FOLDS = 5
SEED = 42

print(f'Device: {DEVICE}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 18.6 MB/s eta 0:00:00
Device: cuda


## 1. Load Data

In [2]:
from set_transformer_10mb import SetTransformerMixture as SetTransformerMixture10mb

ALLELE_MAP = {'X': -2.0, 'Y': -1.0}
SKIP_COLS  = {'TPH', 'PeakCount', 'MaxHeight'}

df = pd.read_csv(CSV_PATH)
locus_cols = [c for c in df.columns
              if c not in ('target_noc', 'group_id')
              and c.rsplit('_', 1)[1] not in SKIP_COLS]
noc_all    = df['target_noc'].values.astype(int)
group_ids  = df['group_id'].values
labels_all = noc_all - 1
vals       = df[locus_cols].values.astype(np.float32)

loci_names = sorted(set(c.rsplit('_', 1)[0] for c in locus_cols))
locus2idx  = {l: i for i, l in enumerate(loci_names)}

col_info = []
for col in locus_cols:
    ln, als = col.rsplit('_', 1)
    li = locus2idx.get(ln, 0)
    av = ALLELE_MAP.get(als, None)
    if av is None:
        try: av = float(als)
        except: av = 0.0
    col_info.append((li, av))

print(f'Dataset: {vals.shape}  loci: {len(loci_names)}  unique groups: {len(np.unique(group_ids))}')
print(f'NOC dist: { {k+1: int((noc_all==k+1).sum()) for k in range(5)} }')

Dataset: (10124, 365)  loci: 24  unique groups: 3418
NOC dist: {1: 8119, 2: 526, 3: 484, 4: 527, 5: 468}


## 2. Build Tokens (8-dim, enriched)

In [3]:
def build_tokens_v4(row_vals, col_info, S=200):
    N = len(row_vals)
    tokens = np.zeros((N, S, 3), dtype=np.float32)
    masks  = np.zeros((N, S),    dtype=np.float32)
    for i in range(N):
        pos = 0
        for j, (li, av) in enumerate(col_info):
            h = row_vals[i, j]
            if h <= 0 or np.isnan(h): continue
            if pos < S:
                tokens[i, pos, 0] = li; tokens[i, pos, 1] = av
                tokens[i, pos, 2] = np.log1p(h); masks[i, pos] = 1.0; pos += 1
    return tokens, masks

def enrich_v4(tokens, masks):
    N, S, _ = tokens.shape
    e = np.zeros((N, S, 8), dtype=np.float32); e[:, :, :3] = tokens
    for i in range(N):
        vi = np.where(masks[i] > 0)[0]
        if len(vi) == 0: continue
        loci  = tokens[i, vi, 0].astype(int); raw_h = np.expm1(tokens[i, vi, 2])
        h_max = raw_h.max() + 1e-9; lp = {}
        for j, idx in enumerate(vi): lp.setdefault(int(loci[j]), []).append((idx, tokens[i, idx, 1], raw_h[j]))
        for l, peaks in lp.items():
            n_l = len(peaks); lsum = sum(p[2] for p in peaks) + 1e-9
            ps  = sorted(peaks, key=lambda x: -x[2]); ah = {round(p[1], 1): p[2] for p in peaks}
            for rank, (idx, av, h) in enumerate(ps):
                ph = ah.get(round(av + 1.0, 1), 0.0)
                e[i, idx, 3] = h / lsum
                e[i, idx, 4] = h / (ph + 1e-9) if ph > 0 else 0.0
                e[i, idx, 5] = rank / max(n_l - 1, 1)
                e[i, idx, 6] = n_l / 12.0
                e[i, idx, 7] = h / h_max
    return e

tok3, msk = build_tokens_v4(vals, col_info)
max_len = int(msk.sum(1).max()) + 2
tok8    = enrich_v4(tok3[:, :max_len, :], msk[:, :max_len])
msk     = msk[:, :max_len]

print(f'Tokens: {tok8.shape}  max_len={max_len}  peaks/sample: {msk.sum(1).mean():.1f}')

Tokens: (10124, 138, 8)  max_len=138  peaks/sample: 44.6


## 3. GroupKFold

In [4]:
gkf = GroupKFold(n_splits=N_FOLDS)
fold_splits = list(gkf.split(tok8, labels_all, groups=group_ids))

for fi, (tr_idx, val_idx) in enumerate(fold_splits, 1):
    dist = Counter(labels_all[val_idx].tolist())
    print(f'  Fold {fi}: val={len(val_idx):4d}  NOC={dict(sorted(dist.items()))}')

  Fold 1: val=2026  NOC={0: 1578, 1: 115, 2: 108, 3: 132, 4: 93}
  Fold 2: val=2025  NOC={0: 1619, 1: 105, 2: 115, 3: 108, 4: 78}
  Fold 3: val=2024  NOC={0: 1587, 1: 132, 2: 87, 3: 107, 4: 111}
  Fold 4: val=2024  NOC={0: 1649, 1: 90, 2: 87, 3: 108, 4: 90}
  Fold 5: val=2025  NOC={0: 1686, 1: 84, 2: 87, 3: 72, 4: 96}


## 4. Model, Loss, Helpers

In [5]:
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0):
        super().__init__()
        self.gamma = gamma; self.weight = weight
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight, reduction='none')
        return ((1 - torch.exp(-ce)) ** self.gamma * ce).mean()


class NOCFinetuneV4(nn.Module):
    def __init__(self, pretrain_sd=None, owner_lut=None, d_model=128, n_noc=5, dropout=0.1):
        super().__init__()
        use_pt = pretrain_sd is not None
        self.backbone = SetTransformerMixture10mb(
            n_loci=24, d_locus=16, d_model=d_model, n_heads=4, n_isab=2, m_inducing=32,
            n_classes=45, n_noc=5, dropout=dropout, cls_decoder='pooled',
            n_token_feats=8, encoder='isab++', num_embed='periodic',
            n_freq=8, d_num_emb=8, periodic_sigma=0.3, nc_attn='mab0',
            feas_filter=use_pt, set_of_set=use_pt,
            owner_lut=owner_lut if use_pt else None,
            aux_heads=use_pt, noc_head_v2=use_pt,
        )
        if use_pt:
            missing, unexpected = self.backbone.load_state_dict(pretrain_sd, strict=False)
            print(f'  Pretrain: loaded {len(pretrain_sd)-len(unexpected)}/{len(pretrain_sd)} tensors')
        self.noc_head = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(d_model, 64),
            nn.ReLU(True), nn.Dropout(dropout), nn.Linear(64, n_noc),
        )

    def forward(self, tokens, mask):
        mb = mask.bool() if mask.dtype != torch.bool else mask
        return self.noc_head(self.backbone.encode(tokens, mb))


def oversample_v4(indices, labels):
    counts = Counter(labels[indices].tolist()); target = max(counts.values()); out = []
    for cls, cnt in counts.items():
        idx = indices[labels[indices] == cls]
        if cnt < target:
            reps = target // cnt; extra = target % cnt
            idx  = np.concatenate([np.tile(idx, reps), np.random.choice(idx, extra, replace=False)])
        out.append(idx)
    return np.concatenate(out)

print('Model + helpers loaded.')

Model + helpers loaded.


In [6]:
def train_fold_v4(model, tok_tr, msk_tr, lbl_tr, tok_val, msk_val, lbl_val,
                  device, epochs=60, lr=3e-4, bs=64, patience=12):
    unique, counts = np.unique(lbl_tr, return_counts=True)
    total = counts.sum()
    cw = torch.tensor([total / (len(unique) * c) for c in counts], dtype=torch.float32).to(device)
    criterion = FocalLoss(weight=cw)

    tr_idx = oversample_v4(np.arange(len(lbl_tr)), lbl_tr); np.random.shuffle(tr_idx)
    dl = DataLoader(TensorDataset(torch.tensor(tok_tr[tr_idx], dtype=torch.float32),
                                   torch.tensor(msk_tr[tr_idx], dtype=torch.float32),
                                   torch.tensor(lbl_tr[tr_idx], dtype=torch.long)),
                    batch_size=bs, shuffle=True, drop_last=True)

    Xv = torch.tensor(tok_val, dtype=torch.float32).to(device)
    Mv = torch.tensor(msk_val, dtype=torch.float32).to(device)
    Yv = torch.tensor(lbl_val, dtype=torch.long)

    for p in model.backbone.parameters(): p.requires_grad = False
    opt = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr*3, weight_decay=1e-4)
    model.train()
    for _ in range(5):
        for xb, mb, yb in dl:
            xb, mb, yb = xb.to(device), mb.to(device), yb.to(device)
            opt.zero_grad(); criterion(model(xb, mb), yb).backward(); opt.step()

    for p in model.backbone.parameters(): p.requires_grad = True
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-6)

    best_macro, best_state, wait = 0.0, None, 0
    for ep in range(epochs):
        model.train(); total_loss = 0
        for xb, mb, yb in dl:
            xb, mb, yb = xb.to(device), mb.to(device), yb.to(device)
            opt.zero_grad()
            loss = criterion(model(xb, mb), yb)
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); total_loss += loss.item()
        sched.step()

        model.eval()
        with torch.no_grad():
            preds = torch.cat([model(Xv[i:i+bs], Mv[i:i+bs]).argmax(1)
                               for i in range(0, len(Xv), bs)]).cpu().numpy()
        macro = f1_score(Yv.numpy(), preds, average='macro')

        if macro > best_macro:
            best_macro = macro; best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}; wait = 0
        else: wait += 1

        if (ep + 1) % 10 == 0 or wait == 0:
            print(f'    Ep {ep+1:3d}: loss={total_loss/len(dl):.4f}  val_macro={macro:.4f}  best={best_macro:.4f}')
        if wait >= patience: print(f'    Early stop ep {ep+1}'); break

    model.load_state_dict(best_state)
    return best_macro

print('train_fold_v4 ready.')

train_fold_v4 ready.


## 5. Load Checkpoint

In [7]:
def load_ckpt(path):
    try:
        with zipfile.ZipFile(path, 'r') as zf:
            buf = io.BytesIO()
            with zipfile.ZipFile(buf, 'w') as out:
                for name in zf.namelist():
                    out.writestr(name, zf.read(name))
            buf.seek(0)
            return torch.load(buf, map_location='cpu', weights_only=False)
    except zipfile.BadZipFile:
        return torch.load(path, map_location='cpu', weights_only=False)

ckpt = load_ckpt(CKPT_PATH)
owner_lut = ckpt.get('owner_lut', None)
print(f'Checkpoint: {len(ckpt)} keys')

Checkpoint: 148 keys


## 6. Run K-Fold

In [8]:
fold_macros = []
all_preds, all_trues = [], []

for fold, (tr_idx, val_idx) in enumerate(fold_splits, 1):
    print(f'\n{"="*50}  Fold {fold}  {"="*50}')
    t0 = time.time()
    torch.manual_seed(SEED + fold); np.random.seed(SEED + fold)

    print(f'  train={len(tr_idx)}  val={len(val_idx)}')
    print(f'  val NOC: {dict(sorted(Counter(labels_all[val_idx].tolist()).items()))}')

    model = NOCFinetuneV4(pretrain_sd=ckpt, owner_lut=owner_lut).to(DEVICE)
    macro = train_fold_v4(model,
                          tok8[tr_idx], msk[tr_idx], labels_all[tr_idx],
                          tok8[val_idx], msk[val_idx], labels_all[val_idx],
                          device=DEVICE, epochs=60, lr=3e-4, bs=64, patience=12)

    model.eval()
    Xv = torch.tensor(tok8[val_idx], dtype=torch.float32).to(DEVICE)
    Mv = torch.tensor(msk[val_idx],  dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        preds = torch.cat([model(Xv[i:i+64], Mv[i:i+64]).argmax(1)
                           for i in range(0, len(Xv), 64)]).cpu().numpy()

    all_preds.append(preds)
    all_trues.append(labels_all[val_idx])
    fold_macros.append(macro)

    print(f'\n  Fold {fold}: Macro={macro:.4f}  ({time.time()-t0:.0f}s)')
    print(classification_report(labels_all[val_idx], preds,
          target_names=[f'NOC={k}' for k in range(1, 6)], digits=3, zero_division=0))

    del model; gc.collect(); torch.cuda.empty_cache()

all_preds = np.concatenate(all_preds)
all_trues = np.concatenate(all_trues)


==================================================  Fold 1  ==================================================
  train=8098  val=2026
  val NOC: {0: 1578, 1: 115, 2: 108, 3: 132, 4: 93}
  Pretrain: loaded 117/148 tensors
    Ep   1: loss=0.9896  val_macro=0.4368  best=0.4368
    Ep   2: loss=0.1545  val_macro=0.7340  best=0.7340
    Ep   3: loss=0.1177  val_macro=0.8353  best=0.8353
    Ep   4: loss=0.0590  val_macro=0.8592  best=0.8592
    Ep   5: loss=0.0673  val_macro=0.8875  best=0.8875
    Ep   7: loss=0.0439  val_macro=0.9048  best=0.9048
    Ep   8: loss=0.0470  val_macro=0.9064  best=0.9064
    Ep  10: loss=0.0362  val_macro=0.8895  best=0.9064
    Ep  11: loss=0.0204  val_macro=0.9142  best=0.9142
    Ep  14: loss=0.0423  val_macro=0.9197  best=0.9197
    Ep  15: loss=0.0285  val_macro=0.9246  best=0.9246
    Ep  16: loss=0.0250  val_macro=0.9430  best=0.9430
    Ep  20: loss=0.0152  val_macro=0.9493  best=0.9493
    Ep  25: loss=0.0061  val_macro=0.9564  best=0.9564
    Ep  

## 7. Results

In [9]:
print(f'DS-ST v4 (noc_10mb) — GroupKFold({N_FOLDS})')
print(f'  Macro-F1: {np.mean(fold_macros):.4f} +/- {np.std(fold_macros):.4f}')
print()

for fi, m in enumerate(fold_macros, 1):
    print(f'  Fold {fi}: {m:.4f}')

# Confusion Matrix (aggregated)
print('\n' + '='*60)
print('NOC Confusion Matrix (aggregated)')
print('='*60)
cm = confusion_matrix(all_trues, all_preds, labels=[0, 1, 2, 3, 4])
cm_df = pd.DataFrame(cm, index=[f'True={k}' for k in range(1, 6)],
                         columns=[f'Pred={k}' for k in range(1, 6)])
print(cm_df.to_string())

cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
cm_pct_df = pd.DataFrame(np.round(cm_pct, 1),
                          index=[f'True={k}' for k in range(1, 6)],
                          columns=[f'Pred={k}' for k in range(1, 6)])
print('\nRow-normalized (%)')
print(cm_pct_df.to_string())

# Save
results = {
    'model': 'DS-ST v4 (noc_10mb)',
    'n_folds': N_FOLDS,
    'macro_f1': f'{np.mean(fold_macros):.4f} +/- {np.std(fold_macros):.4f}',
    'per_fold': fold_macros,
    'confusion_matrix': cm.tolist(),
}
out_path = Path('/kaggle/working/kfold_metrics.json')
out_path.write_text(
    json.dumps(results, indent=2, default=lambda x: float(x) if isinstance(x, np.floating) else str(x)))
print(f'\nSaved -> {out_path}')

DS-ST v4 (noc_10mb) — GroupKFold(5)
  Macro-F1: 0.9436 +/- 0.0158

  Fold 1: 0.9564
  Fold 2: 0.9246
  Fold 3: 0.9241
  Fold 4: 0.9560
  Fold 5: 0.9571

NOC Confusion Matrix (aggregated)
        Pred=1  Pred=2  Pred=3  Pred=4  Pred=5
True=1    8069      28      10       7       5
True=2      25     490       1       2       8
True=3       9       5     458       8       4
True=4       9       3       6     475      34
True=5       5       2       4      12     445

Row-normalized (%)
        Pred=1  Pred=2  Pred=3  Pred=4  Pred=5
True=1    99.4     0.3     0.1     0.1     0.1
True=2     4.8    93.2     0.2     0.4     1.5
True=3     1.9     1.0    94.6     1.7     0.8
True=4     1.7     0.6     1.1    90.1     6.5
True=5     1.1     0.4     0.9     2.6    95.1

Saved -> /kaggle/working/kfold_metrics.json


## 8. Contributor Identification (K-Fold)

Pretrained backbone (slot decoder) → logits_cls → phi deconv (EM) → LOP reranking → top-k decode → Exact Match.
Requires additional dataset: `nguyenmanhhust/kfold-code` (for `SetTransformerMixture`).

In [10]:
import re

# Map NOC predictions back to per-sample array (1-indexed)
noc_pred_all = np.zeros(len(df), dtype=int)
offset = 0
for _, (_, vi) in enumerate(fold_splits):
    n = len(vi)
    noc_pred_all[vi] = all_preds[offset:offset+n] + 1
    offset += n

# Parse donor IDs from group_id filenames
KNOWN_DONORS = [1,2,3,4,5,7,8,9,10,11,12,13,14,15,16,17,18,19,20,
                22,23,24,25,27,28,29,30,31,32,33,34,35,36,37,38,39,
                41,42,43,44,45,46,47,48,49]
DONOR2CLS = {d: i for i, d in enumerate(KNOWN_DONORS)}

def parse_donors(gid, noc):
    parts = gid.split('-')
    if noc == 1:
        m = re.match(r'(\d+)d', parts[2])
        return [int(m.group(1))] if m else []
    return [int(x) for x in parts[2].split('_')]

y_cls = np.zeros((len(df), 45), dtype=np.int32)
valid_cls = np.ones(len(df), dtype=bool)
for i in range(len(df)):
    donors = parse_donors(group_ids[i], noc_all[i])
    if not donors:
        valid_cls[i] = False; continue
    for d in donors:
        if d in DONOR2CLS:
            y_cls[i, DONOR2CLS[d]] = 1
        else:
            valid_cls[i] = False

print(f'Contributor labels: {y_cls.shape}')
print(f'Valid (all known donors): {valid_cls.sum()}/{len(df)}')
print(f'Excluded (unknown donors): {(~valid_cls).sum()}')

Contributor labels: (10124, 45)
Valid (all known donors): 8764/10124
Excluded (unknown donors): 1360


In [11]:
# Clean backbone for contributor ID (needs kfold-code dataset)
sys.path.insert(0, '/kaggle/input/datasets/nguyenmanhhust/kfold-code/code')
from models.set_transformer import SetTransformerMixture

CLS_CFG = dict(n_loci=24, d_locus=16, d_model=128, n_heads=4, n_isab=2, m_inducing=32,
               n_classes=45, dropout=0.1, n_token_feats=8, n_freq=8, d_num_emb=8,
               periodic_sigma=0.3, n_slot_iters=3, ot_eps=0.05, ot_iters=5, gumbel_temp=1.0)

donor_geno = ckpt['donor_geno']
donor_geno_mask = ckpt['donor_geno_mask']
dg_np = donor_geno.cpu().numpy()
dgm_np = donor_geno_mask.cpu().numpy()

# ── phi_rerank (inlined — pure numpy, no model weights) ──
def build_carriers(dg, dgm):
    C = dg.shape[0]; carr = {}
    for c in range(C):
        for j in range(dg.shape[1]):
            if dgm[c, j]:
                key = (int(round(float(dg[c, j, 0]))), int(round(float(dg[c, j, 1]) * 10)))
                carr.setdefault(key, [])
                if c not in carr[key]: carr[key].append(c)
    return carr, C

def deconv_phi(tokens, mask, dg, dgm, n_iters=10):
    carr, C = build_carriers(dg, dgm)
    N = len(tokens); mask = mask.astype(bool)
    PH = np.zeros((N, C), dtype=np.float64)
    for i in range(N):
        idx, keys = [], []
        for p in np.where(mask[i])[0]:
            key = (int(round(float(tokens[i, p, 0]))), int(round(float(tokens[i, p, 1]) * 10)))
            if key in carr: idx.append(p); keys.append(key)
        if not idx: continue
        h = np.expm1(tokens[i, idx, 2].astype(np.float64))
        n = len(idx); S = np.full((n, C + 1), -1e9)
        for r, key in enumerate(keys):
            for c in carr[key]: S[r, c] = 0.0
            S[r, C] = -2.0
        phi = np.ones(C + 1) / (C + 1)
        for _ in range(n_iters):
            z = S + np.log(phi + 1e-9); z -= z.max(1, keepdims=True)
            A = np.exp(z); A /= A.sum(1, keepdims=True)
            w = (A[:, :C] * h[:, None]).sum(0); bg = (A[:, C] * h).sum()
            tot = w.sum() + bg
            phi = np.concatenate([w, [bg]]) / max(tot, 1e-9)
        PH[i] = phi[:C]
    return PH

def _z(a):
    a = np.asarray(a, dtype=np.float64); s = a.std()
    return (a - a.mean()) / (s if s > 1e-9 else 1.0)

def rerank_scores(logits, PH, alpha):
    out = np.empty(logits.shape, dtype=np.float64)
    for i in range(len(logits)):
        out[i] = _z(logits[i]) + alpha * _z(np.log(PH[i] + 1e-6))
    return out

def tune_alpha(L, PH, y, noc, grid=(0.0, 0.2, 0.3, 0.5, 0.75, 1.0)):
    noc = np.clip(noc, 1, 5); C = L.shape[1]
    ks = [k for k in (5, 4, 3) if (noc == k).any()]
    best_a, best_v = 0.0, -1.0
    for a in grid:
        R = rerank_scores(L, PH, a); accs = []
        for k in ks:
            sel = np.where(noc == k)[0]; hit = 0
            for i in sel:
                top = np.argsort(R[i])[::-1][:k]; pr = np.zeros(C, int); pr[top] = 1
                hit += int((pr == y[i]).all())
            accs.append(hit / max(1, len(sel)))
        v = float(np.mean(accs)) if accs else -1.0
        if v > best_v: best_v, best_a = v, a
    return best_a

def topk_decode(scores, k_arr):
    yp = np.zeros_like(scores, dtype=int)
    for i in range(len(scores)):
        k = int(max(1, min(5, round(k_arr[i]))))
        yp[i, np.argsort(scores[i])[::-1][:k]] = 1
    return yp

@torch.no_grad()
def infer_cls(backbone, tok, msk, bs=256):
    backbone.eval(); all_l = []
    for i in range(0, len(tok), bs):
        t = torch.tensor(tok[i:i+bs], dtype=torch.float32).to(DEVICE)
        m = torch.tensor(msk[i:i+bs], dtype=torch.bool).to(DEVICE)
        out = backbone(t, m)
        all_l.append(out['logits_cls'].cpu().numpy())
    return np.concatenate(all_l)

print('Contributor ID helpers ready.')

Contributor ID helpers ready.


In [12]:
print('\n' + '='*60)
print('Contributor Identification — GroupKFold(5)')
print('='*60)

tok3_phi = tok8[:, :, :3].copy()
cls_em_oracle, cls_em_posthoc = [], []

for fold, (tr_idx, val_idx) in enumerate(fold_splits, 1):
    print(f'\n--- Fold {fold} ---')
    t0 = time.time()

    val_v = val_idx[valid_cls[val_idx]]
    tr_v  = tr_idx[valid_cls[tr_idx]]
    print(f'  val: {len(val_idx)} total, {len(val_v)} valid (known donors)')
    if len(val_v) == 0:
        print('  SKIP'); continue

    # Cal: 20% of valid training samples (for alpha tuning)
    rng  = np.random.default_rng(SEED + fold)
    perm = rng.permutation(len(tr_v))
    n_cal = max(len(tr_v) // 5, 100)
    cal_v = tr_v[perm[:n_cal]]

    # Clean backbone (pretrained, NOT NOC-fine-tuned)
    cls_bb = SetTransformerMixture(
        donor_geno=donor_geno, donor_geno_mask=donor_geno_mask,
        owner_lut=owner_lut, **CLS_CFG)
    cls_bb.load_state_dict(ckpt, strict=False)
    cls_bb = cls_bb.to(DEVICE).eval()

    # Forward pass → logits_cls
    L_cal = infer_cls(cls_bb, tok8[cal_v], msk[cal_v])
    L_val = infer_cls(cls_bb, tok8[val_v], msk[val_v])
    del cls_bb; gc.collect(); torch.cuda.empty_cache()

    # Phi deconvolution (independent EM signal)
    PH_cal = deconv_phi(tok3_phi[cal_v], msk[cal_v], dg_np, dgm_np)
    PH_val = deconv_phi(tok3_phi[val_v], msk[val_v], dg_np, dgm_np)

    # Tune alpha on cal set
    alpha = tune_alpha(L_cal, PH_cal, y_cls[cal_v], noc_all[cal_v])
    R_val = rerank_scores(L_val, PH_val, alpha)

    # Top-k decode
    noc_true_v = noc_all[val_v]
    noc_pred_v = noc_pred_all[val_v]

    yp_oracle  = topk_decode(R_val, noc_true_v)
    yp_posthoc = topk_decode(R_val, noc_pred_v)

    em_o = np.all(yp_oracle  == y_cls[val_v], axis=1)
    em_p = np.all(yp_posthoc == y_cls[val_v], axis=1)

    row_o = [em_o.mean()] + [em_o[noc_true_v == k].mean() if (noc_true_v == k).any() else float('nan') for k in range(1, 6)]
    row_p = [em_p.mean()] + [em_p[noc_true_v == k].mean() if (noc_true_v == k).any() else float('nan') for k in range(1, 6)]
    cls_em_oracle.append(row_o)
    cls_em_posthoc.append(row_p)

    print(f'  alpha={alpha:.2f}')
    print(f'  EM oracle:  all={row_o[0]:.3f}  N1={row_o[1]:.3f}  N2={row_o[2]:.3f}  N3={row_o[3]:.3f}  N4={row_o[4]:.3f}  N5={row_o[5]:.3f}')
    print(f'  EM posthoc: all={row_p[0]:.3f}  N1={row_p[1]:.3f}  N2={row_p[2]:.3f}  N3={row_p[3]:.3f}  N4={row_p[4]:.3f}  N5={row_p[5]:.3f}')
    print(f'  ({time.time()-t0:.0f}s)')


Contributor Identification — GroupKFold(5)

--- Fold 1 ---
  val: 2026 total, 1739 valid (known donors)
  alpha=1.00
  EM oracle:  all=0.790  N1=0.854  N2=0.592  N3=0.500  N4=0.491  N5=0.333
  EM posthoc: all=0.786  N1=0.853  N2=0.566  N3=0.479  N4=0.456  N5=0.319
  (5s)

--- Fold 2 ---
  val: 2025 total, 1738 valid (known donors)
  alpha=1.00
  EM oracle:  all=0.807  N1=0.838  N2=0.725  N3=0.670  N4=0.583  N5=0.456
  EM posthoc: all=0.793  N1=0.832  N2=0.706  N3=0.648  N4=0.354  N5=0.456
  (5s)

--- Fold 3 ---
  val: 2024 total, 1752 valid (known donors)
  alpha=1.00
  EM oracle:  all=0.797  N1=0.859  N2=0.533  N3=0.507  N4=0.540  N5=0.414
  EM posthoc: all=0.791  N1=0.856  N2=0.520  N3=0.478  N4=0.460  N5=0.414
  (5s)

--- Fold 4 ---
  val: 2024 total, 1795 valid (known donors)
  alpha=1.00
  EM oracle:  all=0.804  N1=0.843  N2=0.654  N3=0.683  N4=0.542  N5=0.488
  EM posthoc: all=0.799  N1=0.839  N2=0.654  N3=0.651  N4=0.542  N5=0.488
  (5s)

--- Fold 5 ---
  val: 2025 total, 1740 

## 9. Contributor ID Results

In [13]:
em_o = np.array(cls_em_oracle)
em_p = np.array(cls_em_posthoc)
cols = ['All', 'N=1', 'N=2', 'N=3', 'N=4', 'N=5']

print('='*60)
print('Contributor Identification — Summary')
print('='*60)

print('\nEM Oracle (true NOC):')
for fi, row in enumerate(em_o, 1):
    print(f'  Fold {fi}: ' + '  '.join(f'{cols[j]}={row[j]:.3f}' for j in range(len(cols))))
print(f'  Mean:   ' + '  '.join(f'{cols[j]}={np.nanmean(em_o[:, j]):.3f}' for j in range(len(cols))))

print('\nEM Post-hoc (predicted NOC):')
for fi, row in enumerate(em_p, 1):
    print(f'  Fold {fi}: ' + '  '.join(f'{cols[j]}={row[j]:.3f}' for j in range(len(cols))))
print(f'  Mean:   ' + '  '.join(f'{cols[j]}={np.nanmean(em_p[:, j]):.3f}' for j in range(len(cols))))

# Save combined results
results['contributor_id'] = {
    'em_oracle_mean':  {cols[j]: float(np.nanmean(em_o[:, j])) for j in range(len(cols))},
    'em_posthoc_mean': {cols[j]: float(np.nanmean(em_p[:, j])) for j in range(len(cols))},
    'em_oracle_per_fold':  em_o.tolist(),
    'em_posthoc_per_fold': em_p.tolist(),
}
out_path.write_text(
    json.dumps(results, indent=2, default=lambda x: float(x) if isinstance(x, np.floating) else str(x)))
print(f'\nSaved -> {out_path}')

Contributor Identification — Summary

EM Oracle (true NOC):
  Fold 1: All=0.790  N=1=0.854  N=2=0.592  N=3=0.500  N=4=0.491  N=5=0.333
  Fold 2: All=0.807  N=1=0.838  N=2=0.725  N=3=0.670  N=4=0.583  N=5=0.456
  Fold 3: All=0.797  N=1=0.859  N=2=0.533  N=3=0.507  N=4=0.540  N=5=0.414
  Fold 4: All=0.804  N=1=0.843  N=2=0.654  N=3=0.683  N=4=0.542  N=5=0.488
  Fold 5: All=0.826  N=1=0.867  N=2=0.706  N=3=0.561  N=4=0.538  N=5=0.397
  Mean:   All=0.805  N=1=0.852  N=2=0.642  N=3=0.584  N=4=0.539  N=5=0.418

EM Post-hoc (predicted NOC):
  Fold 1: All=0.786  N=1=0.853  N=2=0.566  N=3=0.479  N=4=0.456  N=5=0.319
  Fold 2: All=0.793  N=1=0.832  N=2=0.706  N=3=0.648  N=4=0.354  N=5=0.456
  Fold 3: All=0.791  N=1=0.856  N=2=0.520  N=3=0.478  N=4=0.460  N=5=0.414
  Fold 4: All=0.799  N=1=0.839  N=2=0.654  N=3=0.651  N=4=0.542  N=5=0.488
  Fold 5: All=0.821  N=1=0.863  N=2=0.706  N=3=0.545  N=4=0.487  N=5=0.381
  Mean:   All=0.798  N=1=0.849  N=2=0.630  N=3=0.560  N=4=0.460  N=5=0.412

Saved -> 